In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, accuracy_score

# ====== 1. 讀取資料 ======
def time_to_seconds(tstr):
    if pd.isnull(tstr): return 0
    parts = str(tstr).split(':')
    if len(parts) == 3:
        h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:
        m, s = int(parts[0]), float(parts[1])
        return m * 60 + s
    else:
        try:
            return float(parts[0])
        except:
            return 0

winner = pd.read_csv('./data/winners.csv')
drivers = pd.read_csv('./data/drivers_updated.csv')
teams = pd.read_csv('./data/teams_updated.csv')
laps = pd.read_csv('./data/fastest_laps_updated.csv')

winner['year'] = pd.to_datetime(winner['Date']).dt.year
winner['year_raw'] = winner['year']
winner['Grand Prix raw'] = winner['Grand Prix']

df = winner.merge(
    drivers[['Driver', 'Car', 'year', 'Nationality', 'PTS']],
    left_on=['Winner', 'Car', 'year'],
    right_on=['Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_driver')
)
df = df.merge(
    laps[['Grand Prix', 'Driver', 'Car', 'year', 'Time']],
    left_on=['Grand Prix', 'Winner', 'Car', 'year'],
    right_on=['Grand Prix', 'Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_lap')
)
df = df.merge(
    teams[['Team', 'PTS', 'year']],
    left_on=['Car', 'year'],
    right_on=['Team', 'year'],
    how='left',
    suffixes=('', '_team')
)

df['RaceTime_sec'] = df['Time'].apply(time_to_seconds)
df['FastestLap_sec'] = df['Time_lap'].apply(time_to_seconds)

df['year_raw'] = winner['year_raw']
df['Grand Prix raw'] = winner['Grand Prix raw']

cat_cols = ['Winner', 'Car', 'Grand Prix', 'Nationality', 'Team']
num_cols = [
    'Laps',
    'PTS',        # drivers_updated.csv 的積分
    'PTS_team',   # teams_updated.csv 的積分
    'RaceTime_sec',
    'FastestLap_sec',
    'year'
]
df['PTS_team'] = df['PTS_team'].fillna(0)
df[num_cols] = df[num_cols].fillna(0)

# 過濾只出現一次的冠軍
value_counts = df['Winner'].value_counts()
valid_drivers = value_counts[value_counts >= 2].index
df = df[df['Winner'].isin(valid_drivers)].reset_index(drop=True)

print("過濾前總樣本數：", winner.shape[0])
print("只出現過一次的車手數量：", sum(value_counts == 1))
print("保留後樣本數：", df.shape[0])

# ========== 避免資料洩漏做法：先分割，後fit encoder/scaler ==========
df['orig_index'] = df.index
X_cat = df[cat_cols].astype(str).values
X_num = df[num_cols].values
y = df['Winner'].astype(str).values

X_cat_train, X_cat_test, X_num_train, X_num_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_cat, X_num, y, df['orig_index'].values, test_size=0.2, random_state=42, stratify=y
)


# Encoder fit only on train
encoders = {}
for i, col in enumerate(cat_cols):
    le = LabelEncoder()
    X_cat_train[:, i] = le.fit_transform(X_cat_train[:, i])
    encoders[col] = le

# 篩除測試集未知類別
known_per_col = [set(X_cat_train[:, i]) for i in range(len(cat_cols))]
valid_mask = np.ones(X_cat_test.shape[0], dtype=bool)
for i in range(len(cat_cols)):
    valid_mask &= np.isin(X_cat_test[:, i], list(known_per_col[i]))

X_cat_test = X_cat_test[valid_mask]
X_num_test = X_num_test[valid_mask]
y_test = y_test[valid_mask]
idx_test = idx_test[valid_mask]

# 再做transform
for i, col in enumerate(cat_cols):
    le = encoders[col]
    X_cat_test[:, i] = le.transform(X_cat_test[:, i])

le_winner = encoders['Winner']
y_train_enc = le_winner.transform(y_train)
y_test_enc = le_winner.transform(y_test)

scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_test = scaler.transform(X_num_test)

# ====== PyTorch Dataset ======
class F1RaceSet(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

batch_size = 128
trainset = F1RaceSet(X_cat_train, X_num_train, y_train_enc)
testset = F1RaceSet(X_cat_test, X_num_test, y_test_enc)
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testset, batch_size=batch_size)

# ====== Model ======
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_features, embedding_dim=8, hidden_dim=128, num_classes=None):
        super().__init__()
        self.emb_layers = nn.ModuleList([
            nn.Embedding(cat_dim, embedding_dim) for cat_dim in cat_dims
        ])
        input_dim = embedding_dim * len(cat_dims) + num_num_features
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.emb_layers)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cat_dims = [int(X_cat_train[:,i].max())+1 for i in range(len(cat_cols))]
num_classes = len(le_winner.classes_)

model = F1DNN(cat_dims, len(num_cols), embedding_dim=8, hidden_dim=128, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ====== 訓練流程 ======
epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_cat_batch, X_num_batch, y_batch in train_loader:
        X_cat_batch, X_num_batch, y_batch = X_cat_batch.to(device), X_num_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_cat_batch, X_num_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    avg_loss = total_loss / len(trainset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# ====== 評估與分類報告 ======
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_cat_batch, X_num_batch, y_batch in test_loader:
        X_cat_batch, X_num_batch = X_cat_batch.to(device), X_num_batch.to(device)
        logits = model(X_cat_batch, X_num_batch)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(y_batch.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

unique_y = np.unique(all_labels)
target_names = [f"{idx}: {name}" for idx, name in zip(unique_y, le_winner.inverse_transform(unique_y))]
print(classification_report(
    all_labels, all_preds,
    labels=unique_y,
    target_names=target_names,
    zero_division=0
))

# ====== 準確率 ======
acc = accuracy_score(all_labels, all_preds)
print(f"\nTest Accuracy: {acc:.4f}\n")

# ====== 隨機抽10筆測試集預測結果 ======
test_df = df.iloc[idx_test].reset_index(drop=True)

np.random.seed(42)
rand_idx = np.random.choice(len(test_df), size=10, replace=False)

for i in rand_idx:
    row = test_df.iloc[i]
    year = int(row['year_raw'])
    grand_prix = row['Grand Prix raw']
    true_winner = le_winner.inverse_transform([all_labels[i]])[0]
    pred_winner = le_winner.inverse_transform([all_preds[i]])[0]
    print(f"{year} {grand_prix}\nPredicted Winner: {pred_winner} , True Winner: {true_winner}\n")




過濾前總樣本數： 1110
只出現過一次的車手數量： 36
保留後樣本數： 1074


ValueError: y contains previously unseen labels: 'Jordan Ford'

In [ ]:
import gradio as gr

winners_df = winner  # winner = pd.read_csv('./data/winners.csv')
drivers_df = pd.read_csv('./data/drivers_updated.csv')

# 取得年份與分站選單
year_choices = sorted([int(x) for x in winners_df['year'].dropna().unique()])
grand_prix_choices = sorted([str(x) for x in winners_df['Grand Prix'].dropna().unique()])

def get_driver_combos(year, grand_prix, future_mode=False, future_max_year=None):
    if future_mode:
        # 未來年份只納入近兩年出賽該站的車手
        assert future_max_year is not None
        recent_years = [future_max_year, future_max_year-2]
        df = winners_df[
            (winners_df['Grand Prix'] == grand_prix) &
            (winners_df['year'].isin(recent_years))
        ]
    else:
        # 過去年份 → 只用該年該站有出賽車手
        df = winners_df[(winners_df['year'] == int(year)) & (winners_df['Grand Prix'] == grand_prix)]
    combos = []
    for _, row in df.iterrows():
        driver = str(row['Winner'])
        car = str(row['Car'])
        team = str(row['Car'])  # 或 Team 欄
        nationality = drivers_df[
            (drivers_df['Driver'] == driver) & (drivers_df['year'] == int(row['year']))
        ]['Nationality']
        nationality = nationality.values[0] if not nationality.empty else "UNK"
        combos.append(dict(driver=driver, car=car, team=team, nationality=nationality))
    # 過濾唯一組合
    unique_combos = {}
    for c in combos:
        k = (c['driver'], c['car'], c['team'], c['nationality'])
        unique_combos[k] = c
    return list(unique_combos.values())

def predict_winner(year, grand_prix):
    # 判斷是否為未來年份
    max_data_year = max(year_choices)
    year = int(year)
    is_future = year > max_data_year

    combos = get_driver_combos(
        year, grand_prix, future_mode=is_future, future_max_year=max_data_year
    )
    if not combos:
        return "查無該年該場比賽資料或近兩年無車手出賽，無法預測。"
    probs = []
    for combo in combos:
        cat_inputs = [
            encoders['Winner'].transform([combo['driver']])[0] if combo['driver'] in encoders['Winner'].classes_ else 0,
            encoders['Car'].transform([combo['car']])[0] if combo['car'] in encoders['Car'].classes_ else 0,
            encoders['Grand Prix'].transform([grand_prix])[0] if grand_prix in encoders['Grand Prix'].classes_ else 0,
            encoders['Nationality'].transform([combo['nationality']])[0] if combo['nationality'] in encoders['Nationality'].classes_ else 0,
            encoders['Team'].transform([combo['team']])[0] if combo['team'] in encoders['Team'].classes_ else 0
        ]
        cat_inputs = np.array(cat_inputs).reshape(1, -1)
        num_inputs = [0] * len(num_cols)
        if 'year' in num_cols:
            idx = num_cols.index('year')
            num_inputs[idx] = year
        num_inputs = scaler.transform([num_inputs])
        cat_tensor = torch.tensor(cat_inputs, dtype=torch.long).to(device)
        num_tensor = torch.tensor(num_inputs, dtype=torch.float32).to(device)
        model.eval()
        with torch.no_grad():
            logits = model(cat_tensor, num_tensor)
            prob = torch.softmax(logits, dim=1).cpu().numpy()[0]
            probs.append((prob[cat_inputs[0][0]], combo['driver']))
    probs.sort(reverse=True)
    best_driver = probs[0][1]
    prob_str = "\n".join([f"{d}: {p*100:.2f}%" for p, d in probs[:5]])
    year_desc = "未來預測" if is_future else f"{year}年"
    return f"{year_desc} {grand_prix} 預測最有可能奪冠：{best_driver}\n\n[依機率排序前5名]\n{prob_str}"

with gr.Blocks() as demo:
    year_in = gr.Dropdown(choices=year_choices + [max(year_choices)+1], value=year_choices[-1], label="Year (年份，可選未來)")
    grand_prix_in = gr.Dropdown(choices=grand_prix_choices, value=grand_prix_choices[0], label="Grand Prix (場地/分站)")
    out_box = gr.Textbox(label="預測結果")
    btn = gr.Button("預測")
    btn.click(
        predict_winner,
        inputs=[year_in, grand_prix_in],
        outputs=out_box
    )

demo.launch()


c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with f